# Paper — 01: Orientation Artifact Diagnosis

**Produces:** `figures_paper/fig1_histograms.pdf`, `figures_paper/fig2_synthetic.pdf`

Shows that the 90°/180° spikes in orientation histograms are a rasterization artifact:
1. GT hand-labeled polygons → flat distribution
2. Pixel-aligned mask polygons (YOLO, SAM2) → spiked distribution
3. Synthetic rasterization test → confirms the mechanism deterministically

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Polygon
from shapely import segmentize
from scipy.stats import kstest
from tqdm import tqdm

from rastertools_BOULDERING import metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
prieur_dir      = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px")
prieur_test_dir = prieur_dir / "preprocessing" / "test"
work_dir        = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster       = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

gt_tile_ids = ["1386", "1503", "2054", "2277", "2508"]

# ── Prediction dirs — edit these to match your experiment output dirs ─────
# Each dir should contain *-downscaled-mask-nms.shp files from get_sliced_predictionfast
PRED_CONFIGS = {
    "YOLOv8":          work_dir / "exp_yolo_256",
    "SAM2 zero-shot":  work_dir / "exp_sam2_zero_256",
    "SAM2 fine-tuned": work_dir / "exp_sam2_ft_256",
    "SAM2-auto":       work_dir / "exp_sam2_auto_256",
}
MASK_GLOB = "*-downscaled-mask-nms.shp"

# ── Filters ───────────────────────────────────────────────────────────────
res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)
AR_MIN, AR_MAX  = 1.2, 2.0

# ── Output ────────────────────────────────────────────────────────────────
OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
BINS    = np.linspace(0, 180, 37)   # 5° bins
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "figure.dpi": 150})

print(f"Resolution: {res:.4f} m/px   Areal threshold: {AREAL_THRESHOLD:.4f} m²")

In [ ]:
def run_pipeline(poly, res):
    """segmentize → fitEllipse → MRR → boulder_row. Returns (angle180, aspect_ratio) or None."""
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
    except Exception:
        return None
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
    except Exception:
        return None
    if short_ax < 1e-6:
        return None
    return angle180, long_ax / short_ax


def orientations_from_shapefiles(shp_paths, res, area_thresh=0):
    """Load shapefiles, run full pipeline, return angle180 array for elongated boulders."""
    gdfs = [gpd.read_file(p) for p in shp_paths]
    gdf  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    gdf = gdf[gdf["poly_area"] >= area_thresh].reset_index(drop=True)
    angles = []
    for geom in tqdm(gdf.geometry, leave=False):
        if geom is None or geom.is_empty:
            continue
        result = run_pipeline(geom, res)
        if result and AR_MIN <= result[1] <= AR_MAX:
            angles.append(result[0])
    return np.array(angles)


def make_smooth_ellipse(a, b, theta_deg, cx=0.0, cy=0.0, n_pts=200):
    theta = np.radians(theta_deg)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t) * np.cos(theta) - b * np.sin(t) * np.sin(theta) + cx
    y = a * np.cos(t) * np.sin(theta) + b * np.sin(t) * np.cos(theta) + cy
    return Polygon(np.column_stack([x, y]))


def rasterize_to_pixel_aligned(poly, pix_per_unit=10):
    """Rasterize smooth polygon → extract pixel-aligned contour → Shapely Polygon."""
    minx, miny, maxx, maxy = poly.bounds
    w   = int((maxx - minx) * pix_per_unit) + 20
    h   = int((maxy - miny) * pix_per_unit) + 20
    pad = 5
    pts    = np.array(poly.exterior.coords[:-1])
    pts_px = ((pts - [minx, miny]) * pix_per_unit + pad).astype(np.int32)
    canvas = np.zeros((h, w), np.uint8)
    cv2.fillPoly(canvas, [pts_px], 255)
    cnts, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea).squeeze()
    if cnt.ndim != 2 or len(cnt) < 4:
        return None
    coords = (cnt.astype(float) - pad) / pix_per_unit + [minx, miny]
    return Polygon(coords)


def hist_ax(angles, ax, title, color, gt_angles=None):
    counts, _ = np.histogram(angles, bins=BINS)
    cx = (BINS[:-1] + BINS[1:]) / 2
    ax.bar(cx, counts, width=4.5, color=color, edgecolor="white", lw=0.3)
    if gt_angles is not None:
        gc, _ = np.histogram(gt_angles, bins=BINS)
        ax.step(cx, gc * len(angles) / max(len(gt_angles), 1),
                where="mid", color="seagreen", lw=1.2, label="GT (scaled)")
        ax.legend(fontsize=7)
    ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
    ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180],
           xlabel="Orientation (°)", ylabel="Count")
    ax.set_title(f"{title}\n(n={len(angles)})")
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
print("Loading GT polygons...")
gt_shp_paths = [
    prieur_test_dir / "labels" / f"M1221383405_{tid}_mask.shp"
    for tid in gt_tile_ids
    if (prieur_test_dir / "labels" / f"M1221383405_{tid}_mask.shp").exists()
]
print(f"  Found {len(gt_shp_paths)}/{len(gt_tile_ids)} GT shapefiles")
gt_angles = orientations_from_shapefiles(gt_shp_paths, res)
print(f"  GT elongated boulders (AR {AR_MIN}–{AR_MAX}): {len(gt_angles)}")

In [ ]:
pred_angles = {}
for name, pred_dir in PRED_CONFIGS.items():
    shp_paths = sorted(pred_dir.glob(MASK_GLOB))
    if not shp_paths:
        print(f"  [{name}] No shapefiles in {pred_dir} — skipping")
        continue
    print(f"Loading {name} ({len(shp_paths)} shapefiles)...")
    pred_angles[name] = orientations_from_shapefiles(
        shp_paths, res, area_thresh=AREAL_THRESHOLD)
    print(f"  {len(pred_angles[name])} elongated boulders")

## Figure 1 — Orientation histograms

GT should be flat; all model variants should show 90°/180° spikes.

In [ ]:
all_series = [("GT (Prieur et al.)", gt_angles, "seagreen")] + \
             [(n, a, "#4C72B0") for n, a in pred_angles.items()]

fig, axes = plt.subplots(1, len(all_series), figsize=(2.8 * len(all_series), 2.8))
if len(all_series) == 1:
    axes = [axes]
for ax, (name, angles, color) in zip(axes, all_series):
    hist_ax(angles, ax, name, color)
axes[0].set_facecolor("#f5fbf5")

fig.tight_layout()
fig.savefig(OUT_DIR / "fig1_histograms.pdf", bbox_inches="tight")
plt.show()
print("Saved fig1_histograms.pdf")

In [ ]:
print("KS test vs. uniform — D statistic and p-value:")
for name, angles, _ in all_series:
    D, p = kstest(angles / 180.0, "uniform")
    print(f"  {name:<24}  D={D:.3f}  p={p:.2e}")

## Figure 2 — Synthetic rasterization experiment

Smooth ellipses at known angles → rasterize to pixel-aligned mask → extract polygon → run pipeline.
Expected: smooth polygon recovers correct angle; pixel-aligned polygon snaps to 0°/90°/180°.

Directly analogous to Part 6 of `parts_1_to_6_results.ipynb` but shown as histograms.

In [ ]:
A, B      = 15.0, 10.0   # semi-axes in pipeline coordinate units
RES_SYNTH = 0.5           # segmentize resolution

np.random.seed(42)
N = 300
input_angles = np.random.uniform(0, 180, N)

smooth_recovered, pixalign_recovered = [], []
for theta in tqdm(input_angles, desc="Synthetic test"):
    smooth_poly = make_smooth_ellipse(A, B, theta)
    px_poly     = rasterize_to_pixel_aligned(smooth_poly)

    r_s = run_pipeline(smooth_poly, RES_SYNTH)
    smooth_recovered.append(
        r_s[0] if (r_s and AR_MIN <= r_s[1] <= AR_MAX) else None)

    if px_poly is not None:
        r_p = run_pipeline(px_poly, RES_SYNTH)
        pixalign_recovered.append(
            r_p[0] if (r_p and AR_MIN <= r_p[1] <= AR_MAX) else None)
    else:
        pixalign_recovered.append(None)

valid = [(i, s, p) for i, s, p in zip(input_angles, smooth_recovered, pixalign_recovered)
         if s is not None and p is not None]
in_a  = np.array([v[0] for v in valid])
sm_a  = np.array([v[1] for v in valid])
px_a  = np.array([v[2] for v in valid])
print(f"Valid: {len(valid)}/{N}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))

hist_ax(sm_a, axes[0], "Smooth polygon\n(correct)", "seagreen")
hist_ax(px_a, axes[1], "Pixel-aligned polygon\n(after rasterization)", "firebrick")

axes[2].scatter(in_a, px_a, s=3, alpha=0.4, color="steelblue", rasterized=True)
axes[2].plot([0, 180], [0, 180], "k--", lw=0.7)
axes[2].set(xlabel="Input angle (°)", ylabel="Recovered angle (°)",
            xlim=(0, 180), ylim=(0, 180),
            xticks=[0, 45, 90, 135, 180], yticks=[0, 45, 90, 135, 180])
axes[2].set_title("Input vs. recovered\n(pixel-aligned masks)")
axes[2].spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(OUT_DIR / "fig2_synthetic.pdf", bbox_inches="tight")
plt.show()
print("Saved fig2_synthetic.pdf")